# PDQ White-Box Surrogate Evasion — Analysis & Visualization

**Author:** Avijit Roy  
**Project:** FDEPH — Security Evaluation of Perceptual Image Hashing  
**PI:** Prof. Shweta Jain  
**Date:** March 2026  

---

### What this notebook covers
- PDQ white-box surrogate evasion: 500 ImageNette images × 4 thresholds (T=0.08/0.10/0.12/0.30)
- Per-threshold success rate, step distribution, distortion metrics
- Threshold sweep comparison (matches Phase 1/2 format)
- Cross-method comparison: PDQ vs pHash vs NeuralHash at T=0.10
- Image + hash grid visualization: Original | Adversarial | Diff×10 | Hash grids | XOR
- All figures saved to `fdeph_eval/analysis/figures/`
- All tables saved to `fdeph_eval/analysis/tables/`

**Attack framing:** PDQ white-box surrogate attack, validated against the exact `pdqhash` hard oracle.  
The surrogate approximates PDQ's pipeline in PyTorch for gradient flow; all success events are confirmed by `compute_pdq_hard`.

In [ ]:
import os
import sys
import io
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from PIL import Image

# ---- Repo root --------------------------------------------------------------
HERE = Path(os.getcwd())
REPO_ROOT = HERE
if not (REPO_ROOT / 'fdeph_eval').exists():
    REPO_ROOT = HERE.parent.parent if (HERE.parent.parent / 'fdeph_eval').exists() else HERE
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.pdq_torch import compute_pdq_hard
from utils.image_processing import load_and_preprocess_img

# ---- Paths ------------------------------------------------------------------
LOGS_DIR   = REPO_ROOT / 'logs'
INPUT_DIR  = REPO_ROOT / 'inputs' / 'inputs_500'
FIGURES_DIR = REPO_ROOT / 'fdeph_eval' / 'analysis' / 'figures'
TABLES_DIR  = REPO_ROOT / 'fdeph_eval' / 'analysis' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = ['0.08', '0.10', '0.12', '0.30']
TOTAL_IMAGES = 500
device = 'cuda:0' if __import__('torch').cuda.is_available() else 'cpu'

# ---- Plot style -------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PDQ_COLOR   = '#9C27B0'   # purple — PDQ
NHASH_COLOR = '#2196F3'   # blue   — NeuralHash
PHASH_COLOR = '#FF5722'   # orange — pHash

THRESHOLD_COLORS = {'0.08': '#4CAF50', '0.10': '#2196F3', '0.12': '#FF9800', '0.30': '#E53935'}

print('Repo root:', REPO_ROOT)
print('Device:', device)

## 1. Load All Threshold Logs

In [ ]:
def load_pdq_csv(threshold_str):
    """Load PDQ whitebox sweep CSV for a given threshold.
    Skips __HYPERPARAMS__ row. Returns DataFrame.
    """
    path = LOGS_DIR / f'attack_steps_pdq_whitebox_mt500_T{threshold_str}.csv'
    if not path.exists():
        # Try underscore variant (local upload naming)
        t_under = threshold_str.replace('.', '_')
        path = LOGS_DIR / f'attack_steps_pdq_whitebox_mt500_T{t_under}.csv'
    with open(path) as f:
        content = f.read()
    reader = csv.DictReader(io.StringIO(content))
    rows = []
    for r in reader:
        if r.get('image_id', '').startswith('__'):
            continue
        try:
            rows.append({
                'image_id':       r['image_id'],
                'step':           int(r['step']),
                'elapsed_ms':     float(r['elapsed_ms']) if r['elapsed_ms'] else 0.0,
                'dist_raw':       float(r['dist_raw'])   if r['dist_raw']   else 0.0,
                'dist_norm':      float(r['dist_norm'])  if r['dist_norm']  else 0.0,
                'best_dist_norm': float(r['best_dist_norm']) if r['best_dist_norm'] else 0.0,
                'l2':             float(r['l2'])    if r['l2']    else 0.0,
                'linf':           float(r['linf'])  if r['linf']  else 0.0,
                'ssim':           float(r['ssim'])  if r['ssim']  else 0.0,
                'success':        int(r['success']) if r['success'] else 0,
                'source_path':    r['source_path'],
            })
        except (ValueError, KeyError):
            pass
    return pd.DataFrame(rows)


dfs = {t: load_pdq_csv(t) for t in THRESHOLDS}

for t, df in dfs.items():
    n_img = df['image_id'].nunique()
    n_success = df.groupby('image_id')['success'].max().sum()
    print(f'T={t}: {len(df)} rows, {n_img} images, {int(n_success)}/{TOTAL_IMAGES} succeeded')

## 2. Summary Statistics per Threshold

In [ ]:
def get_success_stats(df):
    """Extract per-image stats at first success step."""
    success_rows = df[df['success'] == 1].copy()
    # Keep first success per image
    first_success = success_rows.sort_values('step').groupby('image_id').first().reset_index()
    return first_success


summary_rows = []
for t in THRESHOLDS:
    df = dfs[t]
    fs = get_success_stats(df)
    n = len(fs)
    summary_rows.append({
        'Threshold':        f'T={t}',
        'Success':          n,
        'Success rate (%)': round(n / TOTAL_IMAGES * 100, 1),
        'Median steps':     int(fs['step'].median()),
        'Mean steps':       round(fs['step'].mean(), 1),
        'P95 steps':        int(fs['step'].quantile(0.95)),
        'Max steps':        int(fs['step'].max()),
        'Median time (ms)': round(fs['elapsed_ms'].median(), 0),
        'Median L2':        round(fs['l2'].median(), 4),
        'Median L-Inf':     round(fs['linf'].median(), 4),
        'Median SSIM':      round(fs['ssim'].median(), 4),
        'SSIM >= 0.99':     int((fs['ssim'] >= 0.99).sum()),
        'SSIM < 0.95':      int((fs['ssim'] < 0.95).sum()),
        'SSIM < 0.90':      int((fs['ssim'] < 0.90).sum()),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Threshold')
display(summary_df.T)

out = TABLES_DIR / 'pdq_whitebox_summary_stats.csv'
summary_df.to_csv(out)
print(f'Saved: {out}')

## 3. Steps Distribution — Histogram per Threshold

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for ax, t in zip(axes, THRESHOLDS):
    fs = get_success_stats(dfs[t])
    color = THRESHOLD_COLORS[t]
    ax.hist(fs['step'], bins=20, color=color, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axvline(fs['step'].median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median = {int(fs["step"].median())}')
    ax.set_title(f'T={t}')
    ax.set_xlabel('Steps to Success')
    ax.set_ylabel('Images')
    ax.legend(fontsize=9)

fig.suptitle('PDQ White-Box Evasion — Steps Distribution (500 images per threshold)', fontsize=13, y=1.02)
plt.tight_layout()
out = FIGURES_DIR / 'pdq_steps_histogram.png'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 4. Cumulative Success Rate vs Steps

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for t in THRESHOLDS:
    fs = get_success_stats(dfs[t])
    sorted_steps = np.sort(fs['step'].values)
    cdf = np.arange(1, len(sorted_steps) + 1) / TOTAL_IMAGES
    ax.plot(sorted_steps, cdf * 100, color=THRESHOLD_COLORS[t],
            linewidth=2, label=f'T={t} ({len(fs)}/500)')

ax.set_xlabel('Optimization Steps')
ax.set_ylabel('Cumulative Success Rate (%)')
ax.set_title('PDQ White-Box Evasion — Cumulative Success Rate vs Steps')
ax.legend()
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
out = FIGURES_DIR / 'pdq_success_rate_vs_steps.png'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 5. Threshold Sweep — Key Metrics Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

t_labels = [f'T={t}' for t in THRESHOLDS]
colors   = [THRESHOLD_COLORS[t] for t in THRESHOLDS]

# Median steps
med_steps = [int(get_success_stats(dfs[t])['step'].median()) for t in THRESHOLDS]
bars = axes[0].bar(t_labels, med_steps, color=colors, alpha=0.85)
axes[0].bar_label(bars, padding=3, fontsize=10)
axes[0].set_title('Median Steps to Success')
axes[0].set_ylabel('Steps')

# Median L-Inf
med_linf = [round(get_success_stats(dfs[t])['linf'].median(), 4) for t in THRESHOLDS]
bars = axes[1].bar(t_labels, med_linf, color=colors, alpha=0.85)
axes[1].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[1].set_title('Median L∞ Distortion')
axes[1].set_ylabel('L∞')

# Median SSIM
med_ssim = [round(get_success_stats(dfs[t])['ssim'].median(), 4) for t in THRESHOLDS]
bars = axes[2].bar(t_labels, med_ssim, color=colors, alpha=0.85)
axes[2].bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
axes[2].set_title('Median SSIM')
axes[2].set_ylabel('SSIM')
axes[2].set_ylim(0.88, 1.0)

fig.suptitle('PDQ White-Box Evasion — Threshold Sweep (500 images)', fontsize=13, y=1.02)
plt.tight_layout()
out = FIGURES_DIR / 'pdq_threshold_sweep_bars.png'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 6. SSIM Distribution

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, t in zip(axes, THRESHOLDS):
    fs = get_success_stats(dfs[t])
    ax.hist(fs['ssim'], bins=20, color=THRESHOLD_COLORS[t], alpha=0.85,
            edgecolor='white', linewidth=0.4)
    ax.axvline(fs['ssim'].median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median={fs["ssim"].median():.4f}')
    ax.axvline(0.99, color='gray', linestyle=':', linewidth=1.0)
    ax.set_title(f'T={t}')
    ax.set_xlabel('SSIM')
    ax.set_ylabel('Images')
    ax.legend(fontsize=8)

fig.suptitle('PDQ White-Box Evasion — SSIM Distribution per Threshold', fontsize=13, y=1.02)
plt.tight_layout()
out = FIGURES_DIR / 'pdq_ssim_distributions.png'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 7. Cross-Method Comparison — PDQ vs pHash vs NeuralHash at T=0.10

In [ ]:
# Reference values from Phase 1 and Phase 2 (T=0.10, 500 images)
# NeuralHash evasion: 37 median steps, 5878ms, L-Inf ~0.04-0.08, SSIM >0.996
# pHash evasion:      13 median steps, 1194ms, L-Inf 0.0078, SSIM 0.9991
pdq_t010 = get_success_stats(dfs['0.10'])

comparison_data = [
    ('NeuralHash\n(96-bit, neural)',  37,                         5878,  0.9991,  '>0.996', NHASH_COLOR),
    ('PDQ\n(256-bit DCT)',            int(pdq_t010['step'].median()), round(pdq_t010['elapsed_ms'].median()), round(pdq_t010['ssim'].median(),4), f'{round(pdq_t010["linf"].median(),4)}', PDQ_COLOR),
    ('pHash\n(64-bit DCT)',           13,                         1194,  0.9991,  '0.0078', PHASH_COLOR),
]

labels = [d[0] for d in comparison_data]
med_steps  = [d[1] for d in comparison_data]
med_time   = [d[2] for d in comparison_data]
med_ssim   = [d[3] for d in comparison_data]
bar_colors = [d[5] for d in comparison_data]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, vals, title, ylabel, fmt in [
    (axes[0], med_steps, 'Median Steps (T=0.10)', 'Steps',    '%d'),
    (axes[1], med_time,  'Median Time ms (T=0.10)', 'ms',     '%d'),
    (axes[2], med_ssim,  'Median SSIM (T=0.10)',  'SSIM',   '%.4f'),
]:
    bars = ax.bar(labels, vals, color=bar_colors, alpha=0.85)
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=10)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

axes[2].set_ylim(0.96, 1.005)
fig.suptitle('Evasion Attack Comparison — NeuralHash vs PDQ vs pHash (T=0.10, 500 images)',
             fontsize=12, y=1.02)
plt.tight_layout()
out = FIGURES_DIR / 'pdq_vs_nhash_vs_phash_comparison.png'
plt.savefig(out, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Also print the table
comp_df = pd.DataFrame({
    'Algorithm':        ['NeuralHash (96-bit neural)', 'PDQ (256-bit DCT)', 'pHash (64-bit DCT)'],
    'Success rate':     ['100%', f"{len(pdq_t010)}/{TOTAL_IMAGES} ({len(pdq_t010)/TOTAL_IMAGES*100:.1f}%)", '100%'],
    'Median steps':     [37, int(pdq_t010['step'].median()), 13],
    'Median time (ms)': [5878, round(pdq_t010['elapsed_ms'].median()), 1194],
    'Median L-Inf':     ['~0.04-0.08', round(pdq_t010['linf'].median(),4), 0.0078],
    'Median SSIM':      ['>0.996', round(pdq_t010['ssim'].median(),4), 0.9991],
}).set_index('Algorithm')
display(comp_df)
comp_df.to_csv(TABLES_DIR / 'pdq_vs_nhash_vs_phash_T010.csv')
print(f'Saved: {TABLES_DIR}/pdq_vs_nhash_vs_phash_T010.csv')

## 8. Image + Hash Grid Visualization
### Original | Adversarial | Diff×10 | Original hash | Adversarial hash | XOR

PDQ hash is 256 bits → displayed as a 16×16 grid. Red cells in XOR grid = flipped bits.

In [ ]:
# ---- Config: which threshold and how many examples to show ------------------
VIZ_THRESHOLD = '0.10'   # change to any of 0.08 / 0.10 / 0.12 / 0.30
N_SHOW = 4

OUT_FOLDER = REPO_ROOT / f'evasion_attack_outputs_pdq_whitebox_T{VIZ_THRESHOLD}'

print(f'Visualizing T={VIZ_THRESHOLD}')
print(f'Looking for adversarial images in: {OUT_FOLDER}')
print(f'Looking for originals in: {INPUT_DIR}')

In [ ]:
# ---- Helper functions -------------------------------------------------------

def find_original(image_id: str, src_dir: Path):
    """Find original image in inputs_500 by image_id stem."""
    for ext in ['.jpeg', '.JPEG', '.jpg', '.png', '.PNG']:
        p = src_dir / f'{image_id}{ext}'
        if p.exists():
            return p
    return None


def find_adversarial(image_id: str, out_dir: Path):
    """Find saved adversarial image in output folder."""
    for suffix in ['_opt_pdq', '_opt']:
        for ext in ['.png', '.PNG', '.jpeg', '.jpg']:
            p = out_dir / f'{image_id}{suffix}{ext}'
            if p.exists():
                return p
    return None


def load_rgb_display(path: Path, size: int = 200) -> np.ndarray:
    return np.array(Image.open(path).convert('RGB').resize((size, size)))


def compute_pdq_grid(path: Path) -> np.ndarray:
    """Compute PDQ hard hash and return as 16x16 numpy grid."""
    tensor = load_and_preprocess_img(str(path), device, resize=True)
    hard_hash, _, _ = compute_pdq_hard(tensor)
    return hard_hash.cpu().numpy().astype(int).reshape(16, 16)


def draw_pdq_hash_grid(ax, grid: np.ndarray, title: str,
                        highlight_mask: np.ndarray = None,
                        color_1='#1565C0', color_0='#E3F2FD',
                        highlight_color='#E53935'):
    """Draw 16×16 PDQ hash grid on axis."""
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 16)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=8, pad=4)
    for r in range(16):
        for c in range(16):
            bit = int(grid[r, c])
            if highlight_mask is not None and highlight_mask[r, c]:
                color, edge, lw = highlight_color, '#8B0000', 0.5
            else:
                color = color_1 if bit == 1 else color_0
                edge, lw = '#90A4AE', 0.2
            ax.add_patch(patches.Rectangle(
                (c, 15 - r), 1, 1,
                linewidth=lw, edgecolor=edge, facecolor=color
            ))

print('Helper functions loaded.')

In [ ]:
# ---- Select diverse examples from the success set ---------------------------
fs_viz = get_success_stats(dfs[VIZ_THRESHOLD]).copy()

# Find pairs that have both original and adversarial saved
paired = []
for _, row in fs_viz.iterrows():
    iid = row['image_id']
    ori = find_original(iid, INPUT_DIR)
    adv = find_adversarial(iid, OUT_FOLDER)
    if ori and adv:
        paired.append({'image_id': iid, 'step': row['step'],
                       'l2': row['l2'], 'linf': row['linf'],
                       'ssim': row['ssim'], 'ori': ori, 'adv': adv})

print(f'Adversarial pairs found: {len(paired)} / {len(fs_viz)}')
if not paired:
    print(f'No adversarial images found in {OUT_FOLDER}.')
    print('Check that the output folder path matches what the sweep runner used.')

In [ ]:
if paired:
    # Pick diverse examples: fastest, best SSIM, median steps, worst SSIM
    paired_df = pd.DataFrame(paired)
    candidates = {
        'Fastest':      paired_df.loc[paired_df['step'].idxmin()],
        'Best SSIM':    paired_df.loc[paired_df['ssim'].idxmax()],
        'Median steps': paired_df.iloc[(paired_df['step'] - paired_df['step'].median()).abs().argsort().iloc[0]],
        'Worst SSIM':   paired_df.loc[paired_df['ssim'].idxmin()],
    }
    # Deduplicate
    seen, examples = set(), {}
    for label, row in candidates.items():
        if row['image_id'] not in seen:
            seen.add(row['image_id'])
            examples[label] = row

    n = len(examples)
    fig, axes = plt.subplots(n, 6, figsize=(20, 3.8 * n))
    if n == 1:
        axes = [axes]

    col_titles = ['Original', 'Adversarial', 'Diff ×10',
                  'Original hash\n(16×16)', 'Adversarial hash\n(flipped=red)', 'XOR\n(red=flipped bits)']

    for row_idx, (example_label, row) in enumerate(examples.items()):
        ori_img = load_rgb_display(row['ori'])
        adv_img = load_rgb_display(row['adv'])
        diff    = np.clip(
            (adv_img.astype(int) - ori_img.astype(int)) * 10 + 128, 0, 255
        ).astype(np.uint8)

        ori_grid = compute_pdq_grid(row['ori'])
        adv_grid = compute_pdq_grid(row['adv'])
        xor_grid = (ori_grid != adv_grid).astype(int)
        n_flipped = int(xor_grid.sum())

        # Images
        for col_idx, img in enumerate([ori_img, adv_img, diff]):
            ax = axes[row_idx][col_idx]
            ax.imshow(img)
            ax.axis('off')
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=9, fontweight='bold')

        # Hash grids
        draw_pdq_hash_grid(axes[row_idx][3], ori_grid, col_titles[3] if row_idx == 0 else '')
        draw_pdq_hash_grid(axes[row_idx][4], adv_grid,
                           col_titles[4] if row_idx == 0 else '',
                           highlight_mask=xor_grid)
        draw_pdq_hash_grid(axes[row_idx][5], xor_grid,
                           col_titles[5] if row_idx == 0 else '',
                           color_1='#E53935', color_0='#E8F5E9',
                           highlight_mask=xor_grid, highlight_color='#E53935')

        # Row label
        axes[row_idx][0].set_ylabel(
            f'{example_label}\n'
            f'steps={int(row["step"])} | {n_flipped} bits flipped\n'
            f'L∞={row["linf"]:.4f} | SSIM={row["ssim"]:.4f}',
            fontsize=8, labelpad=6
        )

    fig.suptitle(
        f'PDQ White-Box Evasion — T={VIZ_THRESHOLD}\n'
        f'Original | Adversarial | Diff×10 | PDQ 16×16 Hash Grids | XOR',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    out = FIGURES_DIR / f'pdq_visualization_T{VIZ_THRESHOLD.replace(".", "")}.png'
    plt.savefig(out, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {out}')

## 9. Artifact Summary

In [ ]:
figures = [
    'pdq_steps_histogram.png',
    'pdq_success_rate_vs_steps.png',
    'pdq_threshold_sweep_bars.png',
    'pdq_ssim_distributions.png',
    'pdq_vs_nhash_vs_phash_comparison.png',
    f'pdq_visualization_T{VIZ_THRESHOLD.replace(".","")}.png',
]
tables = [
    'pdq_whitebox_summary_stats.csv',
    'pdq_vs_nhash_vs_phash_T010.csv',
]

print('=== Figures ===')
for f in figures:
    path = FIGURES_DIR / f
    status = '✅' if path.exists() else '❌ MISSING'
    print(f'  {status}  {path}')

print()
print('=== Tables ===')
for t in tables:
    path = TABLES_DIR / t
    status = '✅' if path.exists() else '❌ MISSING'
    print(f'  {status}  {path}')

print()
print('PDQ analysis complete.')